# Task 5 — archive and publish decision

Every processed event is archived; the email goes out only when the
gates in `auto_tdmt.cfg` §5 pass. The three tables are rebuilt from the
archived `solution.json` files by `catalogue.py`.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
# work in a scratch archive so the real events/ are untouched
os.environ.setdefault("AUTO_TDMT_EVENTS", str(Path.home() / "work" / "proj_tdmt_NZ" / "notebook_runs"))
import config
from config import P            # every tunable, from auto_tdmt.cfg
EVENT = "2026p669681"
print("parameters from", P.source)

In [ ]:
print(json.dumps(P.as_dict()["publish"], indent=2))

## 5.1 The publish decision for an archived solution

In [ ]:
import trigger, okada_forward, publish
cands = sorted(Path(config.REPO_DIR, "events").glob(f"{EVENT}*/solution.json"))
sol = json.loads(cands[0].read_text())
fwd = okada_forward.forward_both_planes(sol)
decision = trigger.publish_decision(sol, fwd, published_history=[])
print(json.dumps(decision, indent=2))
subject, body = publish.draft_text(sol, fwd, sol.get("nisar_passes", []))
print(subject); print(body[:1500])

## 5.2 The tables

In [ ]:
import pandas as pd
cat = pd.read_csv(config.REPO_DIR / "catalogue.csv")
print(cat["Grade"].value_counts().sort_index().to_dict(), "of", len(cat), "events")
display(cat[cat.PublicID == EVENT].T)
npub = pd.read_csv(config.REPO_DIR / "not_published.csv")
print(f"{len(npub)} not published; reasons:")
print(npub["Tags"].value_counts().head(10))
ledger = pd.read_csv(config.REPO_DIR / "station_ledger.csv")
display(ledger[ledger.PublicID == EVENT][["Station", "Distance_km", "Azimuth", "SNR_med", "Used", "Reason_class", "Station_VR", "Shift_s"]])

## What to check
- `not_published.csv` (repo root) says why each event did not email.
- `station_ledger.csv` is the year-one learning table: after a year,
  `station_performance.csv` (its aggregate) shows which stations are
  consistently picked or dropped, and by which rule.